# Cooked Meter — Live Demo Example

Loads the already-trained pipeline (see `notebooks/train_and_evaluate.ipynb`) and runs live predictions on example reviews: raw text + product category in, CS priority + reasoning out. No training here — kept thin on purpose so there's zero risk of long waits or errors during the live presentation.

In [1]:
import os

if os.path.exists('../src'):
    # Running locally from prototype/notebooks/ — just move cwd up to prototype/
    %cd ..
else:
    # Running on a fresh runtime (e.g. Colab) — clone the repo and cd into prototype/
    !git clone https://github.com/Zikabyte/cooked-meter.git
    %cd cooked-meter/prototype

C:\Users\mzida_yqzansz\Projects\notebooks\cooked-meter\prototype


In [2]:
import joblib
import pandas as pd

from src.load_features import clean_review_text

## 1. Load the trained pipeline

All 5 artifacts were saved by `train_and_evaluate.ipynb` — nothing gets fit here.

In [3]:
sentiment_model = joblib.load("models/sentiment_model.pkl")
emotion_model = joblib.load("models/emotion_model.pkl")
reliability_model = joblib.load("models/reliability_model.pkl")
encoder = joblib.load("models/feature_encoder.pkl")
tree_model = joblib.load("models/priority_tree.pkl")

print("Loaded all 5 artifacts.")

Loaded all 5 artifacts.


## 2. The prediction pipeline

`predict_priority` mirrors exactly what `train_and_evaluate.ipynb` does to build a row of `feature_table`, just for one live review instead of a whole dataset: clean the text the same way (`clean_review_text`), run it through Model A (sentiment + emotion) and Model B (reliability), pair that with the product category, then encode + run through Model C.

`explain_decision` walks `tree_model`'s actual decision path for that one row — the exact chain of yes/no splits it followed — so the "reasoning" shown isn't just the 3 predicted features, it's the tree's real logic for this specific prediction.

In [4]:
def explain_decision(x_row):
    """Print the sequence of split conditions tree_model followed to reach
    its prediction for one encoded row."""
    feature_names = encoder.get_feature_names_out()
    x_dense = x_row.toarray()[0] if hasattr(x_row, "toarray") else x_row[0]

    node_indicator = tree_model.decision_path(x_row)
    leaf_id = tree_model.apply(x_row)[0]
    feature = tree_model.tree_.feature
    threshold = tree_model.tree_.threshold

    for node_id in node_indicator.indices:
        if node_id == leaf_id:
            continue
        name = feature_names[feature[node_id]].removeprefix("cat__")
        value = x_dense[feature[node_id]]
        sign = "NO" if value <= threshold[node_id] else "YES"
        print(f"  - {name}? {sign}")


def predict_priority(review_text, product_category):
    """Run one raw review + product category through the full pipeline:
    clean text -> Model A (sentiment, emotion) -> Model B (reliability) ->
    encode -> Model C (priority). Prints the result + reasoning."""
    cleaned = clean_review_text(review_text)

    predicted_sentiment = sentiment_model.predict([cleaned])[0]
    predicted_emotion = emotion_model.predict([cleaned])[0]
    predicted_reliability = reliability_model.predict([cleaned])[0]

    row = pd.DataFrame([{
        "predicted_sentiment": predicted_sentiment,
        "predicted_emotion": predicted_emotion,
        "predicted_reliability": predicted_reliability,
        "product_category": product_category,
    }])
    x_row = encoder.transform(row)
    priority = tree_model.predict(x_row)[0]

    print(f'Review: "{review_text}" ({product_category})')
    print(f"  -> Sentiment: {predicted_sentiment} | Emotion: {predicted_emotion} | Reliability: {predicted_reliability}")
    print(f"  -> Priority: {priority}")
    print("  Reasoning (decision path):")
    explain_decision(x_row)
    print()

    return priority

## 3. Example reviews

A mix on purpose: a clearly urgent complaint, a happy review, a sarcastic one (to show Model B's reliability flag in action), and a mild/neutral one.

In [ ]:
examples = [
    ("Barang rusak pas nyampe, udah komplain berkali-kali gaada respon, kecewa banget sama toko ini!!!", "Electronics"),
    ("Kualitas bagus banget, pengiriman cepat, recommended seller!", "Women's Fashion"),
    ("Seneng banget barangnya cacat. Recommended seller!", "Household"),
    ("Barangnya oke, standar aja sih, sesuai harga.", "Books"),
]

for review_text, product_category in examples:
    predict_priority(review_text, product_category)

Review: "Barang rusak pas nyampe, udah komplain berkali-kali gaada respon, kecewa banget sama toko ini!!!" (Electronics)
  -> Sentiment: Negative | Emotion: Sadness | Reliability: reliable
  -> Priority: Medium
  Reasoning (decision path):
  - predicted_sentiment_Negative? YES
  - predicted_reliability_reliable? YES
  - predicted_emotion_Sadness? YES
  - product_category_Other Products? NO

Review: "Kualitas bagus banget, pengiriman cepat, recommended seller!" (Women's Fashion)
  -> Sentiment: Positive | Emotion: Love | Reliability: reliable
  -> Priority: Low
  Reasoning (decision path):
  - predicted_sentiment_Negative? NO
  - product_category_Camera? NO
  - product_category_Automotive? NO
  - product_category_Kitchen? NO

Review: "Wah mantap sekali ya barangnya sampai dalam kondisi hancur, sangat memuaskan!!!" (Household)
  -> Sentiment: Positive | Emotion: Happy | Reliability: reliable
  -> Priority: Low
  Reasoning (decision path):
  - predicted_sentiment_Negative? NO
  - product_

**Note on example 3 (sarcasm):** this one gets misread as `Low` priority — Model A takes the surface-level positive words ("mantap", "memuaskan") at face value, and Model B doesn't catch it as unreliable either. This is left in deliberately, not fixed: it's an honest, reproducible demonstration of a real limitation (Model B's reliability classifier only has ~54% recall on catching unreliable text — see `train_and_evaluate.ipynb`), directly consistent with the EDA's own finding that sarcasm is one of the hardest linguistic phenomena to detect. Good talking point if judges probe the model's weaknesses.